# BigDebug — breakpoints, watchpoints, and the record that killed the query

Three ways to see inside a running job: stop and look at the state at a point,
guard the records flowing past, and find out which record was in flight when it died.

In [ ]:
import os, sys, glob

ROOT = os.environ.get("BIGASTERISK_HOME") or os.path.abspath("..")

# Jars: a source checkout has them under modules/*/target, the Docker image under jars/.
JARS = sorted(glob.glob(f"{ROOT}/modules/*/target/scala-2.13/bigasterisk-*.jar")) \
    or sorted(glob.glob(f"{ROOT}/jars/bigasterisk-*.jar"))
if not JARS:
    raise SystemExit("No BigAsterisk jars found. Run: bin/sbt package")

FASTUTIL_JAR = os.environ.get("FASTUTIL_JAR") or next(iter(sorted(
    glob.glob(f"{ROOT}/jars/fastutil*.jar")
    + glob.glob(os.path.expanduser("~/Library/Caches/Coursier/**/fastutil-8.5.15.jar"), recursive=True)
    + glob.glob(os.path.expanduser("~/.cache/coursier/**/fastutil-8.5.15.jar"), recursive=True)
)), None)
if not FASTUTIL_JAR:
    raise SystemExit("fastutil jar not found. Run: bin/sbt package")

SPARK_JARS = ",".join(JARS + [FASTUTIL_JAR])
DATA = f"{ROOT}/examples/data"
sys.path.insert(0, f"{ROOT}/python")

## The data

Twelve orders across three customers. One of them, `o8`, is an outlier at
`99999` — every notebook here uses it as the thing to find.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import bigasterisk

spark = (bigasterisk.configure(SparkSession.builder)
    .master("local[2]")
    .appName("bigdebug-notebook")
    .config("spark.jars", SPARK_JARS)
    .config("spark.sql.adaptive.skewJoin.enabled", "false")
    .config("spark.ui.enabled", "false")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

orders = spark.read.schema("oid STRING, cid STRING, amount INT").csv(f"{DATA}/orders.txt")
customers = spark.read.schema("cid STRING, name STRING").csv(f"{DATA}/customers.txt")
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

orders.show()

## A simulated breakpoint

Setting one costs nothing: no operator is inserted and nothing is captured while the query runs. The state is regenerated only when you look.

In [ ]:
bp = bigasterisk.breakpoints(spark).breakpoint(orders.filter("amount > 100"))

# the rest of the query runs at full speed through it
bp.df.groupBy("cid").sum("amount").show()

print(bp.count(), "records were flowing past:")
for row in bp.state(limit=3):
    print(" ", row)

## A watchpoint

A guard on the records flowing past a point. Evaluation is fused into Spark's generated code, so an unmatched row costs one predicate.

In [ ]:
wp = bigasterisk.watchpoints(spark).watch(orders, col("amount") > 1000)
wp.df.groupBy("cid").sum("amount").collect()

print(wp.hits, "record(s) matched", wp.condition)
for row in wp.captured:
    print(" ", row)

## Which record killed the query

Dividing by `(amount - 99999)` is a division by zero for exactly one record, and ANSI mode makes that an error.

In [ ]:
spark.conf.set("spark.sql.ansi.enabled", "true")
guard = bigasterisk.crash_culprit(spark).guard(orders.coalesce(1))
try:
    guard.df.selectExpr("oid", "100 DIV (amount - 99999) AS boom").collect()
except Exception:
    pass
spark.conf.set("spark.sql.ansi.enabled", "false")
print(guard.culprit)

## Check

In [ ]:
assert bp.count() == 8
assert wp.hits == 1 and wp.captured[0]["amount"] == 99999
assert guard.culprit is not None and guard.culprit.row["amount"] == 99999
print("OK")